In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/project-2-intelligent-rag/Starter Notebook.ipynb
/kaggle/input/competitions/project-2-intelligent-rag/sample submission.csv
/kaggle/input/competitions/project-2-intelligent-rag/test.csv
/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus/10_Travel_and_Expense_Policy.pdf
/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus/03_Work_From_Home_Policy.pdf
/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus/09_Onboarding_and_Separation_Policy.pdf
/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus/07_IT_and_Data_Security_Policy.pdf
/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus/06_Compensation_and_Benefits_Policy.pdf
/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus/04_Code_of_Conduct.pdf
/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus/01_Employee_Handbook.pdf
/kaggle/input/competiti

In [2]:
import sys
!{sys.executable} -m pip install -q langchain langchain-community langchain-text-splitters langchain-huggingface langchain-groq langchain-core faiss-cpu pypdf sentence-transformers
print("Installation completed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pa

In [3]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
LLM_API_KEY = user_secrets.get_secret("GROQ_API_KEY")

CORPUS_PATH = "/kaggle/input/competitions/project-2-intelligent-rag/zyro-dynamics-hr-corpus/"
TEST_CSV_PATH = "/kaggle/input/competitions/project-2-intelligent-rag/test.csv"

print("API key loaded:", LLM_API_KEY is not None)

API key loaded: True


In [4]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFDirectoryLoader(CORPUS_PATH)
documents = loader.load()
print(f"Loaded {len(documents)} documents")

/tmp/ipykernel_16/3025785513.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Loaded 39 documents


In [5]:
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Created {len(chunks)} chunks")

Created 107 chunks


In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings_model = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstores = FAISS.from_documents(chunks, embeddings_model)
retriever = vectorstores.as_retriever(search_kwargs={"k": 5})
print("Vector store ready")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store ready


In [7]:
from langchain_groq import ChatGroq

llm_model = ChatGroq(
    model='openai/gpt-oss-20b',
    temperature=0.7,
    max_tokens=500,
    groq_api_key=LLM_API_KEY
)
print("LLM initialized")

LLM initialized


In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT = ChatPromptTemplate.from_template(
    """
    You are an HR assistant. Answer the question using ONLY the context below.
    If the answer is not in the context, say "I don't have that information."

    context : {context},
    question : {question}
    """
)

def format_docs(docs):
    return ".\n\n".join(d.page_content for d in docs)

def rag_chain(question: str):
    docs = retriever.invoke(question)
    context = format_docs(docs)
    chain = RAG_PROMPT | llm_model | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    return {"answer": answer, "sources": docs}

print("RAG chain ready")
 

RAG chain ready


In [9]:
GUARDIAL_PROMPT = ChatPromptTemplate.from_template(
    """
    You are a scope classifier for an HR assistant. Decide whether the question
    below is something an HR assistant should answer (company leave policy,
    reimbursement, code of conduct, etc.). Respond with exactly one word: IN_SCOPE or OUT_OF_SCOPE.

    Question: {question}
    """
)

REFUSAL_MESSAGE = (
    "I'm an HR assistant and can only help with questions about company HR "
    "policies (leave, reimbursement, code of conduct, etc.). I don't have "
    "information to answer that question."
)

def ask_bot(question: str):
    guardial_chain = GUARDIAL_PROMPT | llm_model | StrOutputParser()
    verdict = guardial_chain.invoke({"question": question}).strip().upper()
    if "OUT_OF_SCOPE" in verdict:
        return {"answer": REFUSAL_MESSAGE, "sources": []}
    return rag_chain(question)

print("Guardrail Initialized!")

Guardrail Initialized!


In [10]:
result = ask_bot("How many casual leaves do I get per year?")
print(result["answer"])

print()

result2 = ask_bot("What's the capital of France?")
print(result2["answer"])

The number of casual leave days you receive depends on your grade:

- **Grades L1 to L3** – 30 days per year  
- **Grades L4 to L6** – 60 days per year  
- **Grades L7 to L9** – 90 days per year  
- **Grade L10 (C‑Suite)** – 90 days per year (as per individual contract)

I'm an HR assistant and can only help with questions about company HR policies (leave, reimbursement, code of conduct, etc.). I don't have information to answer that question.


In [11]:
import pandas as pd

test_df = pd.read_csv(TEST_CSV_PATH)

answers = []
for i, row in test_df.iterrows():
    q = row["question"]
    result = ask_bot(q)
    answers.append(result["answer"])
    print(f"[{i+1}/{len(test_df)}] {q}\n-> {result['answer']}\n{'-'*60}")

test_df["answer"] = answers

submission = test_df[["question_id", "answer"]]
submission.to_csv("submission.csv", index=False)
submission.head()

[1/20] How does my Earned Leave accrue every month?
-> Earned Leave (EL) accrues at the following rates:

- **After the first year of continuous service:** 1.25 days per month.  
- **During the probation period:** 0.5 days per month (these days become usable only after probation confirmation).
------------------------------------------------------------
[2/20] How much Earned Leave can I carry forward to next year?
-> You can carry forward a maximum of **45 days** of Earned Leave to the next financial year.
------------------------------------------------------------
[3/20] How many weeks of Maternity Leave am I entitled to?
-> You are entitled to **26 weeks of paid Maternity Leave** for your first two live births.  
If you have a third child, the entitlement is **12 weeks**.  
Additionally, you may take up to **8 weeks of pre‑natal leave** before the expected delivery date.
------------------------------------------------------------
[4/20] Do I need a medical certificate for sick lea

,question_id,answer
0,Q01,Earned Leave (EL) accrues at the following rat...
1,Q02,You can carry forward a maximum of **45 days**...
2,Q03,You are entitled to **26 weeks of paid Materni...
3,Q04,Yes – if your sick leave exceeds 2 consecutive...
4,Q05,Your salary is credited to your registered ban...
